In [ ]:
This notebook demonstrates how to fetch a GLB/GLTF file representing a PC build and integrate it into a Three.js viewer.  You can run the cells sequentially to download the asset and preview it directly in the notebook.</VSCode.Cell>
<VSCode.Cell language="markdown">## 1. Install required packages
We'll use `requests` to download files.  The following cell installs the package if it's not already available.</VSCode.Cell>
<VSCode.Cell language="python"># install the requests library if needed
import sys
import subprocess

try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])
    import requests

print("requests version", requests.__version__)</VSCode.Cell>
<VSCode.Cell language="markdown">## 2. Define download utility
Create a simple helper that takes a URL and a target path and writes the response to disk.</VSCode.Cell>
<VSCode.Cell language="python">import os

def download_file(url, target_path):
    """Download the file at `url` and save it to `target_path`.
    Creates parent directories if necessary."""
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    resp = requests.get(url, stream=True)
    resp.raise_for_status()
    with open(target_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    return target_path

# quick test (URL is not fetched here)
print(download_file.__doc__)</VSCode.Cell>
<VSCode.Cell language="markdown">## 3. Fetch a 3D PC build (GLB)
Use the utility to download a model.  Replace the URL below with a real PC case or full build model if you have one.  For demonstration we'll grab a placeholder GLB from the glTF sample models repository.</VSCode.Cell>
<VSCode.Cell language="python"># choose a model URL and target location
model_url = "https://raw.githubusercontent.com/KhronosGroup/glTF-Sample-Models/master/2.0/Avocado/glTF-Binary/Avocado.glb"
target_file = os.path.join("..", "public", "models", "pc_full.glb")  # relative to notebook

print("downloading to", target_file)
download_file(model_url, target_file)
print("download complete")</VSCode.Cell>
<VSCode.Cell language="markdown">## 4. Verify file exists
Confirm that the download succeeded by checking the filesystem and printing file size.</VSCode.Cell>
<VSCode.Cell language="python">import pathlib

p = pathlib.Path(target_file)
if p.exists():
    print(f"File exists: {p} ({p.stat().st_size} bytes)")
else:
    print("File not found; please check the URL or run the previous cell again.")

print("directory contents:")
for f in p.parent.iterdir():
    print(" -", f.name)  # list files in models directory</VSCode.Cell>
<VSCode.Cell language="markdown">## 5. Embed Three.js viewer in notebook
The following HTML cell sets up a basic Three.js scene, similar to the code used in the PHP application.  It includes the `GLTFLoader` and a canvas for rendering.</VSCode.Cell>
<VSCode.Cell language="python">from IPython.display import HTML

html_template = """
<div id="notebook-pc-wrap" style="width:100%;height:400px;position:relative">
    <canvas id="notebook-pc-canvas" style="width:100%;height:100%;"></canvas>
</div>
<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/examples/js/loaders/GLTFLoader.js"></script>
<script>
(function(){
    var canvas = document.getElementById('notebook-pc-canvas');
    var renderer = new THREE.WebGLRenderer({canvas:canvas,alpha:true,antialias:true});
    renderer.setSize(canvas.clientWidth, canvas.clientHeight);
    var scene = new THREE.Scene();
    var camera = new THREE.PerspectiveCamera(42, canvas.clientWidth/canvas.clientHeight, 0.1, 200);
    camera.position.set(3,1.2,10);
    camera.lookAt(0,0.3,0);
    scene.add(new THREE.AmbientLight(0xffffff,1));
    var key=new THREE.DirectionalLight(0xffffff,1.5); key.position.set(5,10,5); scene.add(key);
    var pc = new THREE.Group(); scene.add(pc);

    var loader = new THREE.GLTFLoader();
    loader.load('pc_full.glb', function(g){
        console.log('model loaded', g);
        pc.add(g.scene);
    }, undefined, function(err){
        console.error('load error', err);
    });

    function animate(){ requestAnimationFrame(animate); renderer.render(scene,camera); }
    animate();
})();
</script>
"""

HTML(html_template)</VSCode.Cell>
<VSCode.Cell language="markdown">## 6. Point viewer at downloaded model
The snippet above tries to load `pc_full.glb` from the same directory as the notebook server's working directory.  If your notebook server is not serving the `public/models` folder, you may need to adjust the path or move the file accordingly.  The downloaded file should now be visible to the viewer.